# LLM4Teach — Reflection Debugger & Experiment Notebook

This notebook is a **debugger, experiment runner, and visualization interface**.
It imports repository modules directly — it does NOT duplicate any logic.

**What this notebook does:**
- Import and configure the training system
- Run short training experiments or evaluate trained models
- Inspect planner prompts and symbolic outputs
- Debug reflection generation and memory contents
- Visualize convergence and episode statistics

**What this notebook does NOT do:**
- Redefine Game, PPO, planner, or controller logic
- Duplicate training loops
- Reimplement any repository module


## 0. Setup — add repo root to path

In [1]:
import sys, os

# Locate repo root regardless of where this notebook was launched from.
# Works when run from notebooks/, repo root, or any other directory.
_this_dir = os.getcwd()
if os.path.basename(_this_dir) == "notebooks":
    REPO_ROOT = os.path.abspath(os.path.join(_this_dir, ".."))
elif os.path.exists(os.path.join(_this_dir, "Game.py")):
    REPO_ROOT = _this_dir
else:
    # Fallback: walk up until we find Game.py
    _d = _this_dir
    while _d != os.path.dirname(_d):
        if os.path.exists(os.path.join(_d, "Game.py")):
            REPO_ROOT = _d
            break
        _d = os.path.dirname(_d)
    else:
        raise RuntimeError("Could not locate LLM4Teach repo root (Game.py not found)")

# Change cwd to repo root so all relative imports and file lookups work
os.chdir(REPO_ROOT)

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("Repo root :", REPO_ROOT)
print("Working dir:", os.getcwd())
print("Python     :", sys.version)


Repo root : c:\Users\HP\Downloads\LLM4Teach-main (1)\LLM4Teach-main
Working dir: c:\Users\HP\Downloads\LLM4Teach-main (1)\LLM4Teach-main
Python     : 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]


## 1. Configure experiment arguments

In [2]:
import argparse
import torch

# Build an args namespace that mirrors main.py's argparse defaults
args = argparse.Namespace(
    # Task
    task          = "SimpleDoorKey",
    frame_stack   = 1,
    offline_planner = True,   # Use pre-computed plans (no LLM needed)
    soft_planner  = False,

    # Training
    seed          = 42,
    seed_list     = [42],
    n_itr         = 50,       # short run for debugging
    traj_per_itr  = 5,
    batch_size    = 64,
    gamma         = 0.99,
    lam           = 0.95,
    recurrent     = False,
    device        = "cuda" if torch.cuda.is_available() else "cpu",

    # Logging
    logdir        = os.path.join(REPO_ROOT, "log"),
    savedir       = "notebook-debug-42",
    loaddir       = None,
    loadmodel     = "acmodel",
    policy        = "ppo",

    # Evaluation
    num_eval      = 5,
    eval_interval = 10,
    save_interval = 25,
    eval_teacher  = False,
    env_seed_list = [0],
)

print("Config:", vars(args))

Config: {'task': 'SimpleDoorKey', 'frame_stack': 1, 'offline_planner': True, 'soft_planner': False, 'seed': 42, 'seed_list': [42], 'n_itr': 50, 'traj_per_itr': 5, 'batch_size': 64, 'gamma': 0.99, 'lam': 0.95, 'recurrent': False, 'device': 'cpu', 'logdir': 'c:\\Users\\HP\\Downloads\\LLM4Teach-main (1)\\LLM4Teach-main\\log', 'savedir': 'notebook-debug-42', 'loaddir': None, 'loadmodel': 'acmodel', 'policy': 'ppo', 'num_eval': 5, 'eval_interval': 10, 'save_interval': 25, 'eval_teacher': False, 'env_seed_list': [0]}


## 2. Instantiate Game and optionally attach reflection

In [3]:
from Game import Game

game = Game(args)
print("Game created.")
print("  Task:", args.task)
print("  Obs space:", game.obs_space)
print("  Action space:", game.action_space)
print("  Max episode length:", game.max_ep_len)

[INFO]: resetting the task: SimpleDoorKey
Logging to c:\Users\HP\Downloads\LLM4Teach-main (1)\LLM4Teach-main\log\ppo\SimpleDoorKey\notebook-debug-42
use MLP......
Game created.
  Task: SimpleDoorKey
  Obs space: {'image': (10, 10, 4), 'text': 100}
  Action space: 7
  Max episode length: 150


In [4]:
# ── Optional: attach Qwen planner (online planning) ──────────────────────────
# Uncomment to use online Qwen planning instead of offline cached plans.
# Requires Ollama running with qwen2.5:3b pulled.

# from utils.qwen_llm import QwenLLM
# qwen_planner = QwenLLM(backend='ollama', model='qwen2.5:3b')
# game.teacher_policy.planner.set_llm(qwen_planner)
# print("Qwen planner attached.")

In [5]:
# ── Optional: attach reflection system ───────────────────────────────────────
# Uncomment to enable reflection. Requires online planner (above).

# from memory.reflection import QwenReflector
# from memory.memory_buffer import ReflectionMemory
#
# reflector = QwenReflector(backend='ollama', model='qwen2.5:7b')
# memory    = ReflectionMemory(maxlen=20, top_k=5)
# game.set_reflection_system(reflector, memory)
# print("Reflection system attached.")

## 3. Inspect planner — single-step debug

In [6]:
import numpy as np

# Reset environment and teacher policy, get a single observation
obs = game.env.reset()
game.teacher_policy.reset()

# Get text representation of obs
planner = game.teacher_policy.planner
obs_text = planner.mediator.RL2LLM(obs[0])
print("Observation text:", obs_text)

Observation text: Agent sees <key>, holds <nothing>.


In [7]:
# Get plans and probabilities for this observation
plans, probs = planner.plan(obs_text)
print("Plans:", plans)
print("Probs:", [round(p, 3) for p in probs])

Plans: ['go to <key>, pick up <key>', 'pick up <key>']
Probs: [0.98, 0.02]


In [8]:
# Full teacher policy step
teacher_probs = game.teacher_policy(obs[0])
action_names = ["turn left", "turn right", "forward", "pick up", "drop", "toggle"]
for i, (name, prob) in enumerate(zip(action_names, teacher_probs)):
    bar = "█" * int(prob * 30)
    print(f"  {name:12s} | {prob:.3f} | {bar}")

  turn left    | 0.000 | 
  turn right   | 0.000 | 
  forward      | 1.000 | ██████████████████████████████
  pick up      | 0.000 | 
  drop         | 0.000 | 
  toggle       | 0.000 | 


## 4. Symbolic parser validation

In [9]:
from utils.symbolic_parser import strict_parse, validate_plan

test_plans = [
    # Valid plans
    "go to <key>, pick up <key>",
    "explore",
    "go to <door>, open <door>",
    "drop <key>, go to <red door>, open <red door>",
    
    # Invalid plans (should be rejected)
    "walk to key",             # no <> wrapper
    "go to (7,3)",             # coordinate
    "pick up the blue thing",  # not wrapped in <>
    "",                        # empty
]

print("Strict parser validation:")
print("-" * 60)
for plan in test_plans:
    valid, errors = validate_plan(plan)
    result = strict_parse(plan)
    status = "✅" if result else "❌"
    print(f"{status} Input:  '{plan}'")
    if result:
        print(f"   Output: '{result}'")
    else:
        print(f"   Errors: {errors}")
    print()

Strict parser validation:
------------------------------------------------------------
✅ Input:  'go to <key>, pick up <key>'
   Output: 'go to <key>, pick up <key>'

✅ Input:  'explore'
   Output: 'explore'

✅ Input:  'go to <door>, open <door>'
   Output: 'go to <door>, open <door>'

✅ Input:  'drop <key>, go to <red door>, open <red door>'
   Output: 'drop <key>, go to <red door>, open <red door>'

[SymbolicParser] ✗ Invalid token 'walk to key': missing <object> reference in token: 'walk to key'
[SymbolicParser] ✗ Plan 'walk to key' produced no valid tokens → reject
❌ Input:  'walk to key'
   Errors: ["missing <object> reference in token: 'walk to key'"]

[SymbolicParser] ✗ Invalid token 'go to (7': coordinates not allowed in plan token: 'go to (7'
[SymbolicParser] ✗ Invalid token '3)': coordinates not allowed in plan token: '3)'
[SymbolicParser] ✗ Plan 'go to (7,3)' produced no valid tokens → reject
❌ Input:  'go to (7,3)'
   Errors: ["coordinates not allowed in plan token: 'go to 

## 5. Reflection system — manual test

In [10]:
from memory.reflection import EpisodeTrajectory, validate_reflection, clean_reflection

# Build a synthetic trajectory for testing the reflector
traj = EpisodeTrajectory()
traj.add_step("Agent sees <nothing>, holds <nothing>.", "explore", 0.0)
traj.add_step("Agent sees <key>, holds <nothing>.",    "go to <key>, pick up <key>", 0.0)
traj.add_step("Agent sees <door>, holds <key>.",       "go to <door>, open <door>",  1.0)
traj.finish(success=True)

print("Trajectory prompt:")
print("-" * 50)
print(traj.to_prompt())

Trajectory prompt:
--------------------------------------------------
Episode result: SUCCESS
Episode length: 3 steps
Total reward: 1.00

Unique observations (most frequent first):
  - Agent sees <nothing>, holds <nothing>.  (x1)
  - Agent sees <key>, holds <nothing>.  (x1)
  - Agent sees <door>, holds <key>.  (x1)

Plans used (most frequent first):
  - explore  (x1)
  - go to <key>, pick up <key>  (x1)
  - go to <door>, open <door>  (x1)

Write a 1-3 sentence strategic reflection on this episode.
Summarize ONLY observed facts. Do NOT speculate or invent entities.


In [11]:
# Test reflection validation
test_reflections = [
    # Good
    "Exploring early revealed the key quickly. Once the key was obtained, the door was opened efficiently.",
    # Bad — coordinate
    "The key was found at position (7,3). Then the door was at (4,2).",
    # Bad — speculation
    "There was probably a hidden room behind the wall.",
    # Bad — fabricated object
    "The bookshelf contained a key near the emergency exit.",
]

print("Reflection validation:")
for r in test_reflections:
    ok = validate_reflection(r)
    status = "✅ VALID" if ok else "❌ REJECTED"
    print(f"{status}: {r[:70]}")

Reflection validation:
✅ VALID: Exploring early revealed the key quickly. Once the key was obtained, t
❌ REJECTED: The key was found at position (7,3). Then the door was at (4,2).
❌ REJECTED: There was probably a hidden room behind the wall.
❌ REJECTED: The bookshelf contained a key near the emergency exit.


In [12]:
# Test reflector with offline backend (always returns None — no LLM call)
from memory.reflection import QwenReflector

reflector = QwenReflector(backend='offline')
result = reflector.reflect(traj)
print("Offline reflector result:", result)  # Expected: None
print("Stats:", reflector.stats)

# To test with Ollama:
# reflector = QwenReflector(backend='ollama', model='qwen2.5:7b')
# result = reflector.reflect(traj)
# print("Reflection:", result)

Offline reflector result: None
Stats: {'generated': 0, 'validated': 0, 'rejected': 0, 'failed': 0, 'skipped_empty': 0}


## 6. Reflection memory — add, retrieve, and inspect

In [13]:
from memory.memory_buffer import ReflectionMemory

memory = ReflectionMemory(maxlen=10, top_k=3)

# Add some synthetic reflections
memory.add_memory(
    "Exploring room boundaries early revealed the blue key faster.",
    episode_id=1, success=True, ep_len=45, total_reward=1.0
)
memory.add_memory(
    "Revisiting already-explored regions wasted steps without finding the key.",
    episode_id=2, success=False, ep_len=128, total_reward=0.0
)
memory.add_memory(
    "After picking up the key, going directly to the door without extra exploration succeeded.",
    episode_id=3, success=True, ep_len=38, total_reward=1.0
)

print("Memory size:", len(memory))
print(memory.summary())

Memory size: 3
ReflectionMemory: 3/10 entries | added=3 dup=0 filtered=0 success=2 failure=1


In [14]:
# Inspect the context string that would be injected into the planner
context = memory.get_context()
print("Planner context injection:")
print("-" * 50)
print(context)

Planner context injection:
--------------------------------------------------
Strategic memory from past episodes:
  [success] After picking up the key, going directly to the door without extra exploration succeeded.
  [failure] Revisiting already-explored regions wasted steps without finding the key.
  [success] Exploring room boundaries early revealed the blue key faster.


In [15]:
# Test deduplication
added = memory.add_memory(
    "Exploring room boundaries early revealed the blue key faster.",
    episode_id=10, success=True
)
print("Duplicate added?", added)  # Expected: False
print(memory.summary())

Duplicate added? False
ReflectionMemory: 3/10 entries | added=3 dup=1 filtered=0 success=2 failure=1


In [16]:
# Save and reload
save_path = os.path.join(REPO_ROOT, "log", "debug_memory.json")
memory.save(save_path)

memory2 = ReflectionMemory(maxlen=10, top_k=3)
memory2.load(save_path)
print("Reloaded memory size:", len(memory2))
print(memory2.get_context())

[ReflectionMemory] Saved 3 entries → c:\Users\HP\Downloads\LLM4Teach-main (1)\LLM4Teach-main\log\debug_memory.json
[ReflectionMemory] Loaded 3 entries from c:\Users\HP\Downloads\LLM4Teach-main (1)\LLM4Teach-main\log\debug_memory.json
Reloaded memory size: 3
Strategic memory from past episodes:
  [success] After picking up the key, going directly to the door without extra exploration succeeded.
  [failure] Revisiting already-explored regions wasted steps without finding the key.
  [success] Exploring room boundaries early revealed the blue key faster.


## 7. Short training run

In [17]:
# Run a short training experiment (offline planner — no LLM needed)
# Adjust n_itr in the args cell above to control training length

# Re-create game to start fresh
game_train = Game(args)
print(f"Starting {args.n_itr}-iteration training run...")
game_train.train()
print("Training complete.")

[INFO]: resetting the task: SimpleDoorKey
Logging to c:\Users\HP\Downloads\LLM4Teach-main (1)\LLM4Teach-main\log\ppo\SimpleDoorKey\notebook-debug-42
use MLP......
Starting 50-iteration training run...
********** Iteration 0 ************
time elapsed: 0.00 s
2.69 s to collect    750 timesteps | 279.12sample/s.
0.97 s to optimizer| loss 13.033, entropy  1.235, kickstarting  1.272.
-------------------------------------------------
|                 Timesteps |             750 |
|            Return (train) |             0.0 |
|    Episode Length (train) |           150.0 |
|      Success Rate (train) |             0.0 |
-------------------------------------------------
********** Iteration 1 ************
time elapsed: 3.66 s
2.73 s to collect    750 timesteps | 274.57sample/s.
0.79 s to optimizer| loss 12.913, entropy  1.239, kickstarting  1.278.
-------------------------------------------------
|                 Timesteps |            1500 |
|            Return (train) |             0.0 |

## 8. Visualize training results

In [18]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from simulator.visualize_training import (
    plot_reward_curve,
    plot_success_rate,
    plot_ppo_losses,
    plot_reflection_stats,
    plot_training_overview,
)

# Point to the run log directory
run_log_dir = game_train.logger.dir
print("Log dir:", run_log_dir)

Log dir: c:\Users\HP\Downloads\LLM4Teach-main (1)\LLM4Teach-main\log\ppo\SimpleDoorKey\notebook-debug-42


In [19]:
fig = plot_training_overview(run_log_dir)
if fig:
    import matplotlib.pyplot as plt
    plt.show()
else:
    print("No plot data yet — run training first.")

C:\Users\HP\AppData\Local\Temp\ipykernel_13016\4000237061.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
fig = plot_ppo_losses(run_log_dir)
if fig:
    plt.show()

C:\Users\HP\AppData\Local\Temp\ipykernel_13016\738313376.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
fig = plot_reflection_stats(run_log_dir)
if fig:
    plt.show()

C:\Users\HP\AppData\Local\Temp\ipykernel_13016\175091538.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Evaluate trained model

In [22]:
import numpy as np

# Evaluate student policy (deterministic)
n_eval = 10
returns, lengths, successes = [], [], []

for _ in range(n_eval):
    ret, length, success = game_train.evaluate(deterministic=True, record_frames=False)
    returns.append(ret)
    lengths.append(length)
    successes.append(success)

print(f"Evaluation over {n_eval} episodes:")
print(f"  Mean return:      {np.mean(returns):.3f}")
print(f"  Mean ep length:   {np.mean(lengths):.1f}")
print(f"  Success rate:     {np.mean(successes):.2%}")

Evaluation over 10 episodes:
  Mean return:      0.000
  Mean ep length:   150.0
  Success rate:     0.00%


## 10. Inspect planner dictionary (offline cache)

In [23]:
planner = game_train.teacher_policy.planner

print(f"Plans cache has {len(planner.plans_dict)} entries:")
print("-" * 60)
for obs_text, (plans, probs) in planner.plans_dict.items():
    print(f"OBS: {obs_text}")
    for plan, prob in zip(plans, probs):
        bar = "█" * int(prob * 20)
        print(f"  {prob:.2f} {bar} → {plan}")
    print()

Plans cache has 6 entries:
------------------------------------------------------------
OBS: Agent sees <nothing>, holds <nothing>.
  1.00 ████████████████████ → explore

OBS: Agent sees <door>, holds <nothing>.
  1.00 ████████████████████ → explore

OBS: Agent sees <key>, holds <nothing>.
  0.98 ███████████████████ → go to <key>, pick up <key>
  0.02  → pick up <key>

OBS: Agent sees <nothing>, holds <key>.
  0.68 █████████████ → explore
  0.22 ████ → go to <door>, open <door>
  0.04  → explore, go to <door>, open <door>
  0.02  → explore, go to <door>
  0.02  → explore, open <door>
  0.02  → go to <door>, pick up <handle>, use <key>

OBS: Agent sees <door>, holds <key>.
  0.62 ████████████ → go to <door>, open <door> with <key>
  0.30 ██████ → go to <door>, open <door>
  0.06 █ → go to <key>, pick up <key>, go to <door>, open <door>
  0.02  → explore, go to <door>

OBS: Agent sees <key>, <door>, holds <nothing>.
  0.84 ████████████████ → go to <key>, pick up <key>, go to <door>, open

## 11. End-to-end reflection pipeline demo

In [24]:
# Full pipeline demonstration:
# 1. Build a trajectory from a real episode
# 2. Generate a reflection (offline → returns None; replace with Ollama for live test)
# 3. Store in memory
# 4. Show planner prompt with injected context

from memory.reflection import QwenReflector, EpisodeTrajectory
from memory.memory_buffer import ReflectionMemory

demo_game = Game(args)
demo_memory = ReflectionMemory(maxlen=10, top_k=3)

# Add a synthetic reflection manually (normally generated by QwenReflector)
demo_memory.add_memory(
    "Exploring the perimeter found the key in fewer steps than random search.",
    episode_id=0, success=True, ep_len=50, total_reward=1.0
)

# Wire memory into the planner
demo_game.teacher_policy.planner.set_reflection_memory(demo_memory)

# Now show what prompt would be sent to the LLM
obs = demo_game.env.reset()
obs_text = demo_game.teacher_policy.planner.mediator.RL2LLM(obs[0])
augmented = demo_game.teacher_policy.planner._build_prompt_with_reflection(obs_text)

print("Observation text (cache key, unchanged):")
print(" ", obs_text)
print()
print("Augmented LLM prompt (with reflection context):")
print("-" * 60)
print(augmented)

[INFO]: resetting the task: SimpleDoorKey
Logging to c:\Users\HP\Downloads\LLM4Teach-main (1)\LLM4Teach-main\log\ppo\SimpleDoorKey\notebook-debug-42
use MLP......
Observation text (cache key, unchanged):
  Agent sees <nothing>, holds <nothing>.

Augmented LLM prompt (with reflection context):
------------------------------------------------------------
Strategic memory from past episodes:
  [success] Exploring the perimeter found the key in fewer steps than random search.

Current observation:
Agent sees <nothing>, holds <nothing>.
